In [2]:
import sqlite3
import pandas as pd

# Why sqlite3: it's built into Python, no installation needed
# We're creating a local database file in our project folder
db_path = r"C:\Users\likhi\OneDrive\Desktop\e commerce project\outputs\olist.db"
conn = sqlite3.connect(db_path)

print("Database connected successfully")

Database connected successfully


In [3]:
# Load the clean CSV we made in Day 2
clean_path = r"C:\Users\likhi\OneDrive\Desktop\e commerce project\outputs\clean_orders.csv"
df = pd.read_csv(clean_path)

# Why to_sql: this writes the dataframe as a table inside our SQLite database
# if_exists='replace' means if table already exists, overwrite it
df.to_sql('orders', conn, if_exists='replace', index=False)

print(f"Table created successfully")
print(f"Rows loaded: {len(df)}")

Table created successfully
Rows loaded: 110832


In [4]:
query1 = """
SELECT 
    customer_state,
    COUNT(*) as total_orders,
    SUM(CASE WHEN late_delivery = 1 THEN 1 ELSE 0 END) as late_orders,
    ROUND(100.0 * SUM(CASE WHEN late_delivery = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) as late_rate_pct,
    ROUND(AVG(review_score), 2) as avg_review_score,
    ROUND(AVG(delivery_days), 1) as avg_delivery_days
FROM orders
GROUP BY customer_state
HAVING total_orders > 100
ORDER BY late_rate_pct DESC
"""

result1 = pd.read_sql_query(query1, conn)
print(result1.head(10))

  customer_state  total_orders  late_orders  late_rate_pct  avg_review_score  \
0             AL           431          104           24.1              3.82   
1             MA           805          163           20.2              3.77   
2             SE           375           61           16.3              3.90   
3             PI           524           81           15.5              3.96   
4             CE          1431          219           15.3              3.87   
5             BA          3703          508           13.7              3.86   
6             RJ         14224         1846           13.0              3.87   
7             TO           310           38           12.3              4.16   
8             PA          1061          131           12.3              3.84   
9             ES          2236          273           12.2              4.02   

   avg_delivery_days  
0               24.0  
1               21.1  
2               21.0  
3               18.9  
4   

In [5]:
query2 = """
SELECT 
    product_category_name_english as category,
    COUNT(*) as total_orders,
    ROUND(SUM(price), 2) as total_revenue,
    ROUND(AVG(price), 2) as avg_order_value,
    ROUND(AVG(review_score), 2) as avg_review_score
FROM orders
WHERE product_category_name_english != 'unknown'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 15
"""

result2 = pd.read_sql_query(query2, conn)
print(result2)

                 category  total_orders  total_revenue  avg_order_value  \
0           health_beauty          9519     1237439.95           130.00   
1           watches_gifts          5867     1166968.63           198.90   
2          bed_bath_table         11107     1037177.69            93.38   
3          sports_leisure          8488      960010.09           113.10   
4   computers_accessories          7707      896132.29           116.28   
5         furniture_decor          8239      718344.78            87.19   
6              housewares          6819      617836.73            90.61   
7              cool_stuff          3727      612071.86           164.23   
8                    auto          4157      580146.14           139.56   
9                    toys          4037      471920.79           116.90   
10           garden_tools          4282      471671.27           110.15   
11                   baby          2990      400774.42           134.04   
12              perfumery

In [6]:
query3 = """
SELECT 
    product_category_name_english as category,
    COUNT(*) as total_orders,
    ROUND(SUM(price), 2) as total_revenue,
    ROUND(AVG(review_score), 2) as avg_review_score,
    ROUND(AVG(delivery_days), 1) as avg_delivery_days,
    ROUND(AVG(CASE WHEN late_delivery = 1 THEN 1.0 ELSE 0 END) * 100, 1) as late_rate_pct
FROM orders
WHERE product_category_name_english != 'unknown'
GROUP BY category
ORDER BY late_rate_pct DESC
LIMIT 15
"""

result3 = pd.read_sql_query(query3, conn)
print(result3)

                             category  total_orders  total_revenue  \
0                      home_comfort_2            30         760.27   
1   furniture_mattress_and_upholstery            37        4323.38   
2                               audio           363       50620.50   
3             fashion_underwear_beach           127        9305.95   
4                  christmas_supplies           150        8737.84   
5                     books_technical           265       18755.20   
6                        home_confort           432       58265.15   
7           construction_tools_lights           302       40185.00   
8                                food           499       28731.15   
9                         electronics          2730      155173.83   
10                      health_beauty          9519     1237439.95   
11                   office_furniture          1678      269418.10   
12                               baby          2990      400774.42   
13                mu

In [7]:
query4 = """
SELECT 
    order_month,
    COUNT(DISTINCT order_id) as total_orders,
    ROUND(SUM(price), 2) as total_revenue,
    ROUND(AVG(price), 2) as avg_order_value
FROM orders
GROUP BY order_month
ORDER BY order_month
"""

result4 = pd.read_sql_query(query4, conn)
print(result4)

   order_month  total_orders  total_revenue  avg_order_value
0      2016-09             1         134.97            44.99
1      2016-10           265       40451.80           127.61
2      2016-12             1          10.90            10.90
3      2017-01           750      112573.39           121.83
4      2017-02          1653      235483.40           125.99
5      2017-03          2546      360865.25           123.71
6      2017-04          2303      341209.62           132.41
7      2017-05          3545      492441.44           121.68
8      2017-06          3135      425387.66           120.71
9      2017-07          3872      484637.74           108.66
10     2017-08          4193      559317.01           115.30
11     2017-09          4150      609750.15           127.80
12     2017-10          4478      651438.50           123.94
13     2017-11          7288      995082.64           116.57
14     2017-12          5513      728836.17           117.29
15     2018-01          

In [8]:
query5 = """
SELECT 
    seller_id,
    COUNT(DISTINCT order_id) as total_orders,
    ROUND(SUM(price), 2) as total_revenue,
    ROUND(AVG(price), 2) as avg_order_value,
    ROUND(AVG(review_score), 2) as avg_review_score
FROM orders
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 15
"""

result5 = pd.read_sql_query(query5, conn)
print(result5)

                           seller_id  total_orders  total_revenue  \
0   4869f7a5dfa277a7dca6462dcf3b52b2          1124      226987.93   
1   53243585a1d6dc2643021fd1853d8905           348      217940.44   
2   4a3ca9315b744ce9f8e9374361493884          1772      199408.32   
3   fa1c13f2614d7b5c4749cbc52fecda94           578      190917.14   
4   7c67e1448b00f6e969d365cea6b010ab           973      188063.83   
5   7e93a43ef30c4f03f38b393420bc753a           319      165981.49   
6   da8622b14eb17ae2831f4ac5b9dab84a          1311      162303.67   
7   7a67c85e85bb2ce8582c35f2203ad736          1145      140238.65   
8   1025f0e2d44d7041d6cf58b6550e0bfa           910      139720.16   
9   955fee9216a65b617aa5c0531780ce60          1261      131906.71   
10  46dc3b2cc0980fb8ec44634e21d2718e           503      122811.38   
11  6560211a19b47992c3666cc44a7e94c0          1819      120983.82   
12  620c87c171fb2a6dd6e8bb4dec959fc6           722      112970.90   
13  7d13fca15225358621be4086e1eb09

In [9]:
query6 = """
SELECT 
    review_score,
    COUNT(*) as total_orders,
    ROUND(AVG(delivery_days), 1) as avg_delivery_days,
    ROUND(AVG(CASE WHEN late_delivery = 1 THEN 1.0 ELSE 0 END) * 100, 1) as late_rate_pct
FROM orders
WHERE review_score IS NOT NULL
GROUP BY review_score
ORDER BY review_score DESC
"""

result6 = pd.read_sql_query(query6, conn)
print(result6)

   review_score  total_orders  avg_delivery_days  late_rate_pct
0           5.0         63305               10.2            3.0
1           4.0         21184               11.8            4.9
2           3.0          9242               13.6           10.3
3           2.0          3700               15.3           17.9
4           1.0         12574               19.1           31.9


In [10]:
queries = {
    "q1_late_delivery_by_state.sql": query1,
    "q2_revenue_by_category.sql": query2,
    "q3_delivery_by_category.sql": query3,
    "q4_monthly_revenue.sql": query4,
    "q5_top_sellers.sql": query5,
    "q6_review_vs_delivery.sql": query6
}

sql_path = r"C:\Users\likhi\OneDrive\Desktop\e commerce project\sql\\"

for filename, query in queries.items():
    with open(sql_path + filename, 'w') as f:
        f.write(query.strip())
    print(f"Saved: {filename}")

print("\nAll SQL files saved!")

Saved: q1_late_delivery_by_state.sql
Saved: q2_revenue_by_category.sql
Saved: q3_delivery_by_category.sql
Saved: q4_monthly_revenue.sql
Saved: q5_top_sellers.sql
Saved: q6_review_vs_delivery.sql

All SQL files saved!
